<font size="5"><center>***Gestión de riesgos financieros (parte III).***</center></font>

<font size="4"><center>***Master en Ingeniería Matemática.***</center></font>

<font size="5"><center><span style="color:blue">***Nombre y Apellidos: Ana Marta Oliveira dos Santos, Maria Albarrán Sanchéz, Siria Catherine Íñiguez Brito***</span></center></font>

In [ ]:
nombreyapellidos = "Nombre Apellido1 Apellido2"
if nombreyapellidos == "":
    print("Rellena tu nombre completo!")
else:
    print("Gracias!: ", nombreyapellidos, ":)")

Gracias!:  Nombre Apellido1 Apellido2 :)


<font size="5"><center><span style="color:blue">***Prácticas GRF (parte III) 7,8,9***</span></center></font>

### Práctica 7: Calcula el VaR y ES 95% y 99% sobre una acción.

Las penúltimas elecciones presidenciales en Estados Unidos se celebraron el martes 3 de noviembre de 2020.   

El 7 de noviembre, cuatro días después del día de la elección, el candidato demócrata Joe Biden fue anunciado como el virtual ganador de las elecciones presidenciales, a la espera de los cómputos finales y posterior ratificación de los resultados por parte de los diferentes estados.

El día 14 de diciembre Biden fue oficialmente elegido presidente por parte del Colegio Electoral.   

La elección del Colegio Electoral fue ratificada por el Senado el día 6 de enero de 2021.

En el fichero adjunto a esta práctica de GRF dispones de los datos de Open, Close, High, Low, Adjusted Close relativo a los siguientes subyacentes:

<img src="Valores GRF 2021.png" style="height: 300px">
<center style="color:#888"><br/></center>

In [ ]:
import numpy as np
import pandas as pd
from datetime import timedelta

df = pd.read_pickle("historico_desde_2020_trabajo_GRF.pkl")


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 279 entries, 2019-12-31 to 2021-01-28
Columns: 150 entries, ('Adj Close', 'ACS.MC') to ('Volume', 'VIS.MC')
dtypes: float64(150)
memory usage: 329.1 KB


In [ ]:
df.columns

MultiIndex([('Adj Close',  'ACS.MC'),
            ('Adj Close',     'AMD'),
            ('Adj Close',    'AMZN'),
            ('Adj Close',    'ATVI'),
            ('Adj Close', 'BBVA.MC'),
            ('Adj Close',    'CSCO'),
            ('Adj Close',     'DIS'),
            ('Adj Close',      'EA'),
            ('Adj Close',    'EBAY'),
            ('Adj Close',     'FOX'),
            ...
            (   'Volume',    'MXIM'),
            (   'Volume',    'NFLX'),
            (   'Volume',    'NVDA'),
            (   'Volume',  'SAB.MC'),
            (   'Volume',  'SAN.MC'),
            (   'Volume',    'SBUX'),
            (   'Volume', 'SGRE.MC'),
            (   'Volume',  'TEF.MC'),
            (   'Volume',    'TSLA'),
            (   'Volume',  'VIS.MC')],
           length=150)

In [ ]:
df2 = df['Adj Close']
df2.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 279 entries, 2019-12-31 to 2021-01-28
Data columns (total 25 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   ACS.MC   274 non-null    float64
 1   AMD      271 non-null    float64
 2   AMZN     271 non-null    float64
 3   ATVI     271 non-null    float64
 4   BBVA.MC  275 non-null    float64
 5   CSCO     271 non-null    float64
 6   DIS      271 non-null    float64
 7   EA       271 non-null    float64
 8   EBAY     271 non-null    float64
 9   FOX      271 non-null    float64
 10  IDR.MC   274 non-null    float64
 11  ITX.MC   274 non-null    float64
 12  MAP.MC   274 non-null    float64
 13  MEL.MC   274 non-null    float64
 14  MRL.MC   274 non-null    float64
 15  MXIM     271 non-null    float64
 16  NFLX     271 non-null    float64
 17  NVDA     271 non-null    float64
 18  SAB.MC   274 non-null    float64
 19  SAN.MC   276 non-null    float64
 20  SBUX     271 non-null    float64
 2

###### Dispones de 24.000 euros (aproximadamente, considera la cifra que prefieras) que puedes repartir en la forma que desees sobre el portfolio que definas sobre los valores anteriores.

-Periodo **pre-Elecciones** (sugerencia 45 días naturales anteriores a las elecciones 3-nov-2.020 pero elige el periodo que consideres).  
-Periodo **pos-Elecciones** (sugerencia 45 días naturales posteriores a las elecciones, de nuevo elige el periodo que desees).
Podeis revisar días de sesiones activas (sesiones bursátiles).

#### **1. Realiza una comparativa del VaR y del ES sobre un portfolio con una sola acción en el periodo preelectoral y poselectoral.**

Para analizar el impacto de las elecciones presidenciales de EE. UU. de 2020 en el riesgo de mercado, hemos definido tres escenarios independientes. Se han construido tres carteras individuales para observar cómo la incertidumbre política afectó a diferentes perfiles de empresas.

Para cada caso, se ha considerado una inversión inicial de **24.000 €** y se ha comparado el comportamiento en una ventana de 45 días naturales antes y después de la jornada electoral del 3 de noviembre de 2020.

Los activos seleccionados para esta comparativa son:
* **Netflix (NFLX):** Empresa clave del sector de tecnología de consumo y servicios de streaming.
* **Amazon (AMZN):** Referente del comercio electrónico y servicios en la nube.
* **Disney (DIS):** Representante del sector entretenimiento y ocio.

El cálculo de las métricas se ha realizado mediante el método de **simulación histórica**, utilizando rendimientos logarítmicos y un nivel de confianza del **95%**.

In [ ]:
# 1. Definir parámetros globales
inversion_inicial = 24000
confianza = 0.95
fecha_elecciones = pd.to_datetime("2020-11-03")
ventana_dias = 45
activos = ['NFLX', 'AMZN', 'DIS']

# Definir periodos (45 días naturales)
inicio_pre = fecha_elecciones - timedelta(days=ventana_dias)
fin_pre = fecha_elecciones - timedelta(days=1)
inicio_pos = fecha_elecciones + timedelta(days=1)
fin_pos = fecha_elecciones + timedelta(days=ventana_dias)

# 2. Función para calcular VaR y ES (Método Histórico)
def calcular_metricas_riesgo(rendimientos_serie, inversion, alpha=0.95):
    if rendimientos_serie.empty:
        return np.nan, np.nan

    # VaR: Cuantil (1-alpha)
    var_percentil = np.percentile(rendimientos_serie, (1 - alpha) * 100)
    var_euros = inversion * var_percentil

    # ES: Media de los rendimientos peores que el VaR
    peores_rendimientos = rendimientos_serie[rendimientos_serie <= var_percentil]
    es_euros = inversion * peores_rendimientos.mean()

    return var_euros, es_euros

# 3. Bucle para procesar cada activo
resultados_lista = []

for activo in activos:
    # Extraer precios y calcular rendimientos logarítmicos
    try:
        precios = df[('Close', activo)].dropna()
        rendimientos = np.log(precios / precios.shift(1)).dropna()

        # Filtrar por periodos
        rend_pre = rendimientos.loc[inicio_pre:fin_pre]
        rend_pos = rendimientos.loc[inicio_pos:fin_pos]

        # Calcular métricas
        var_pre, es_pre = calcular_metricas_riesgo(rend_pre, inversion_inicial, confianza)
        var_pos, es_pos = calcular_metricas_riesgo(rend_pos, inversion_inicial, confianza)

        # Guardar resultados
        resultados_lista.append({
            'Activo': activo,
            'VaR Pre (€)': f"{var_pre:.2f} €",
            'VaR Pos (€)': f"{var_pos:.2f} €",
            'ES Pre (€)': f"{es_pre:.2f} €",
            'ES Pos (€)': f"{es_pos:.2f} €"
        })
    except KeyError:
        print(f"Error: No se encontraron datos para {activo} en el DataFrame.")

# 4. Mostrar resultados consolidados
df_final = pd.DataFrame(resultados_lista)

print(f"Comparativa de Riesgo por Activo (Inversión: {inversion_inicial}€)")
print("="*80)
print(df_final.to_string(index=False))


Comparativa de Riesgo por Activo (Inversión: 24000€)
Activo VaR Pre (€) VaR Pos (€) ES Pre (€) ES Pos (€)
  NFLX  -1266.96 €   -727.14 € -1558.25 € -1532.81 €
  AMZN   -965.58 €   -687.31 € -1177.68 € -1045.73 €
   DIS   -784.52 €   -553.20 €  -888.45 €  -814.47 €


Tras procesar los datos históricos de **Netflix**, **Amazon** y **Disney**, hemos obtenido una visión cuantitativa del riesgo antes y después del hito electoral. A continuación, explicamos qué significan estos indicadores y qué nos dicen sobre el mercado en 2020:

1. **VaR 95% (Value at Risk):** Esta métrica nos indica la pérdida máxima esperada en un horizonte de un día, con un nivel de confianza del 95%.
   *  En el periodo pre-elecciones, el VaR de **NFLX** fue de **-1.266,96 €**. Esto significa que, en condiciones normales de mercado, hay un 95% de probabilidades de que no pierdas más de esa cantidad en un solo día. Solo en el 5% de los casos (los peores días) la pérdida superaría esa cifra.

2. **ES 95% (Expected Shortfall / CVaR):**
   A diferencia del VaR, el ES nos dice qué sucede cuando "atravesamos" la barrera del 5% de peores escenarios. Es la media de todas las pérdidas que exceden el VaR.
   * Mientras el VaR pone un límite, el ES mide la **severidad de la catástrofe**. Es una métrica mucho más conservadora y realista para la gestión de riesgos, ya que considera la "cola" de la distribución de rendimientos.


### Evaluación Crítica de las Métricas de Riesgo: VaR vs. Expected Shortfall

El análisis de las carteras individuales de **24.000 €** permite contrastar la eficacia del **Value at Risk (VaR)** y el **Expected Shortfall (ES)** bajo el escenario de estrés de las elecciones de 2020. A continuación, se evalúan sus ventajas y limitaciones técnicas utilizando los resultados obtenidos como evidencia empírica.

---

#### **A. Value at Risk (VaR): El umbral de confianza y la volatilidad de mercado**
El VaR se define como el cuantil de la distribución que marca el límite de pérdida máxima esperada. En este estudio, ha demostrado ser un indicador altamente sensible a la **estabilización del sentimiento de mercado**.

* **Evidencia: El Efecto "Resolución de Incertidumbre"** Tras la jornada electoral, el mercado reaccionó positivamente a la reducción de la incertidumbre política. Esto se refleja en la contracción del **42.6%** en el VaR de **Netflix (NFLX)** (de -1.266,96 € a -727,14 €). A los mercados financieros no les gusta la incertidumbre, antes del 3 de noviembre, la volatilidad era alta por el desconocimiento del resultado. Una vez que el panorama político se aclaró, los precios se estabilizaron, lo que redujo la pérdida potencial diaria.
* **Pros (Ventajas):**
    * **Estandarización y Comunicación:** Facilita la síntesis del riesgo en un valor monetario único, simplificando la gestión de límites en entornos corporativos.
    * **Capacidad de Backtesting:** Permite validar el modelo mediante el **recuento de excepciones**. Para un nivel de confianza del 95%, se esperan aproximadamente 12-13 fallos anuales; un exceso de excepciones indicaría que el modelo está infravalorando el riesgo.
* **Contras (Limitaciones):**
    * **Invisibilidad del riesgo de cola:** Es una métrica "ciega" ante la magnitud de las pérdidas una vez superado el umbral. No diferencia entre una pérdida ligeramente superior al VaR y una catástrofe financiera.
    * **Falta de subaditividad:** Matemáticamente, no es una métrica coherente de riesgo, lo que puede derivar en una infravaloración del riesgo al no capturar adecuadamente los beneficios de la diversificación en distribuciones no normales.

#### **B. Expected Shortfall (ES): La medida de la severidad y el riesgo sistémico**
El ES calcula la esperanza matemática de las pérdidas que exceden el VaR, ofreciendo una visión mucho más conservadora y realista de la "cola" de la distribución.

* **Evidencia: La Persistencia de Riesgos Extremos (Colas Pesadas)** El hallazgo técnico más crítico es la divergencia observada en **NFLX**: mientras su VaR se redujo con fuerza, su **ES apenas varió** (-1.558,25 € vs -1.532,81 €). Esto demuestra que, aunque los eventos negativos se volvieron menos frecuentes tras las elecciones, su **severidad e intensidad** permanecieron constantes. El riesgo de movimientos bruscos de pánico en el sector tecnológico no desapareció con el resultado electoral.
* **Pros (Ventajas):**
    * **Coherencia Matemática:** Cumple con la propiedad de subaditividad, lo que garantiza que el modelo siempre reconozca los beneficios de la diversificación de carteras.
    * **Sensibilidad a "Fat Tails":** Es indispensable para activos de alta volatilidad, ya que cuantifica cuánto se perdería, de media, en los peores escenarios posibles.

    
* **Contras (Limitaciones):**
    * **Complejidad de validación:** Su backtesting es significativamente más difícil de implementar que el del VaR y requiere series temporales de datos más extensas para garantizar robustez estadística.
    * **Inestabilidad ante Outliers:** Al promediar exclusivamente los valores extremos, un solo dato atípico en la muestra puede sesgar la métrica de forma desproporcionada.


---


La comparación entre los activos seleccionados revela la necesidad de emplear ambas métricas de forma complementaria según el perfil del activo:

1.  **Disney (DIS) como Activo de Refugio:** Presentó los niveles de riesgo más bajos y una mayor proximidad entre VaR y ES. Esta **convergencia** sugiere una distribución de retornos más cercana a la normalidad, posicionando a Disney como el activo más resiliente ante la incertidumbre política. Además,  su menor ES indica que sus caídas extremas son menos severas que las de las "Big Tech".
2.  **Netflix (NFLX) y la Divergencia Técnica:** La amplia brecha entre su VaR y su ES evidencia una distribución con **colas pesadas**. Aquí, el VaR por sí solo daría una falsa sensación de seguridad tras las elecciones, mientras que el ES revela que el riesgo extremo sigue latente.

Para una gestión de riesgos de nivel institucional, el **VaR** es la herramienta óptima para el control diario de límites y consumo de capital, mientras que el **Expected Shortfall** es la métrica técnica superior para la planificación de contingencias y la evaluación de la resiliencia ante crisis de liquidez o eventos catastróficos.

##### **2. Realiza una comparativa del VaR considerando retornos simples y otra con retornos logarítmicos**

En la gestión de riesgos financieros, la elección del tipo de retorno es fundamental. Mientras que los **retornos simples** representan fielmente la variación porcentual de la inversión, los **retornos logarítmicos** (o continuamente compuestos) ofrecen propiedades matemáticas superiores para el modelado estadístico, como la aditividad temporal.

En este apartado, evaluamos si la elección del método de cálculo genera diferencias significativas en la estimación del riesgo para una inversión de **24.000 €** en el periodo pre-electoral.

Para que la comparativa sea justa, hemos aplicado las siguientes fórmulas:

* **Rendimiento Simple:** $R_s = \frac{P_t - P_{t-1}}{P_{t-1}}$
* **Rendimiento Logarítmico:** $R_{log} = \ln\left(\frac{P_t}{P_{t-1}}\right)$
* **Conversión del VaR Logarítmico:** Dado que el logaritmo comprime los valores negativos, para obtener la pérdida real en euros hemos aplicado la transformación: $VaR_{\text{euros}} = \text{Inversión} \times (e^{VaR_{log}} - 1)$.

In [ ]:
# 1. Parámetros de la práctica
inversion = 24000
confianza = 0.95
fecha_elecciones = pd.to_datetime("2020-11-03")
inicio_pre = fecha_elecciones - timedelta(days=45)
fin_pre = fecha_elecciones - timedelta(days=1)

# Acciones a analizar
activos = ['NFLX', 'AMZN', 'DIS']

# Lista para almacenar los resultados
resultados_comparativa = []

# 2. Iterar sobre cada activo
for activo in activos:
    try:
        # Extraer precios de la acción y filtrar para el periodo pre-electoral
        precios = df[('Adj Close', activo)].dropna()
        precios_pre = precios.loc[inicio_pre:fin_pre]

        # Comprobar que hay suficientes datos en el periodo
        if len(precios_pre) > 1:
            # Cálculo de Rendimientos Simples: (P_t / P_{t-1}) - 1
            rend_simples = precios_pre.pct_change().dropna()

            # Cálculo de Rendimientos Logarítmicos: ln(P_t / P_{t-1})
            rend_log = np.log(precios_pre / precios_pre.shift(1)).dropna()

            # Cálculo del VaR Simple (al 95%)
            var_percentil_simple = np.percentile(rend_simples, (1 - confianza) * 100)
            var_euro_simple = inversion * var_percentil_simple

            # Cálculo del VaR Logarítmico (al 95%)
            var_percentil_log = np.percentile(rend_log, (1 - confianza) * 100)
            var_euro_log = inversion * (np.exp(var_percentil_log) - 1)

            # Diferencia absoluta entre ambos métodos
            dif_relativa = abs(var_euro_simple - var_euro_log)

            # Guardar métricas en la lista
            resultados_comparativa.append({
                'Activo': activo,
                'Percentil Simple (5%)': f"{var_percentil_simple:.4%}",
                'Percentil Log (5%)': f"{var_percentil_log:.4%}",
                'VaR Simple (€)': f"{var_euro_simple:.2f} €",
                'VaR Log (€)': f"{var_euro_log:.2f} €",
                'Diferencia Abs. (€)': f"{dif_relativa:.4f} €"
            })
        else:
            print(f"Advertencia: No hay datos suficientes en el periodo pre-electoral para {activo}.")

    except KeyError:
        print(f"Error: No se encontraron datos para {activo} en el DataFrame.")

# 3. Mostrar resultados consolidados
comparativa_df = pd.DataFrame(resultados_comparativa)

print("Comparativa de VaR: Retornos Simples vs Logarítmicos (Periodo Pre-Electoral)")
print("-" * 90)
print(comparativa_df.to_string(index=False))


Comparativa de VaR: Retornos Simples vs Logarítmicos (Periodo Pre-Electoral)
------------------------------------------------------------------------------------------
Activo Percentil Simple (5%) Percentil Log (5%) VaR Simple (€) VaR Log (€) Diferencia Abs. (€)
  NFLX              -5.1913%           -5.3323%     -1245.91 €  -1246.23 €            0.3203 €
  AMZN              -3.9616%           -4.0424%      -950.78 €   -950.82 €            0.0417 €
   DIS              -3.2286%           -3.2819%      -774.85 €   -774.87 €            0.0196 €


Al observar los datos obtenidos para nuestros tres activos, podemos extraer las siguientes conclusiones técnicas:

* 1. **La Divergencia en los Percentiles**
Es notable que el **Percentil Log (5%)** siempre es un número más negativo que el **Percentil Simple (5%)**. Por ejemplo, en **NFLX**, pasamos de un -5.19% a un -5.33%.
  Esto ocurre por la propia naturaleza de la función logarítmica, que "penaliza" más las caídas. Matemáticamente, para cualquier caída de precio, el retorno logarítmico siempre será menor (más extremo) que el simple.

* 2. **Convergencia en el Valor Monetario (VaR €)**
A pesar de la diferencia en los porcentajes, cuando convertimos ambos valores a euros (aplicando la función exponencial al logarítmico), los resultados son casi idénticos:
  * En **Disney (DIS)**, la diferencia es de apenas **0.0196 €** (menos de 2 céntimos).
  * En **Netflix (NFLX)**, que es el activo más volátil, la diferencia máxima es de solo **0.32 €**.


* 3. **Conclusión sobre la Precisión**
La comparativa demuestra que, para **horizontes temporales diarios** y volatilidades estándar, ambos métodos son igualmente válidos para la toma de decisiones.
  * **El rendimiento simple** es directo y no requiere transformaciones adicionales para entender la pérdida de caja.
  * **El rendimiento logarítmico**, aunque requiere la fórmula $V_0 \times (e^r - 1)$, es el preferido si decidiéramos escalar este VaR a una semana o un mes, ya que permite sumar los rendimientos diarios sin errores de capitalización.


### Práctica 8: Calcula el VaR y ES 95% y 99% sobre una cartera de acciones.

#### **3. Realiza una comparativa del VaR sobre un portfolio con las 3 acciones que decidas en el periodo preelectoral y poselectoral.**  

In [ ]:
from scipy.stats import norm
np.random.default_rng(12345)

# 1. Configuración Inicial
inversion_total = 24000
pesos = np.array([1/3, 1/3, 1/3])  # Portfolio equitativo
confianza = 0.95
z_alpha = norm.ppf(1 - confianza)  # Valor crítico z
n_simulaciones = 10000
fecha_elecciones = pd.to_datetime("2020-11-03")
ventana_dias = 45
tickers = ['NFLX', 'AMZN', 'DIS']

# 2. Preparación de Datos y Rendimientos
precios = df['Adj Close'][tickers].dropna()
rend_activos = np.log(precios / precios.shift(1)).dropna()

# Definición de periodos
inicio_pre, fin_pre = fecha_elecciones - timedelta(days=ventana_dias), fecha_elecciones - timedelta(days=1)
inicio_pos, fin_pos = fecha_elecciones + timedelta(days=1), fecha_elecciones + timedelta(days=ventana_dias)

def calcular_todo_el_riesgo(rendimientos, inversion, weights, n_sim):
    # -- MÉTODO 1: PARAMÉTRICO (VARIANZA-COVARIANZA)
    # Supone normalidad y usa la matriz de covarianza
    matriz_cov = rendimientos.cov()
    mu_activos = rendimientos.mean()

    # Riesgo de la cartera: sigma_c = sqrt(w' * Sigma * w)
    sigma_portfolio = np.sqrt(weights.T @ matriz_cov @ weights)
    mu_portfolio = np.sum(mu_activos * weights)

    var_param = inversion * (-mu_portfolio + abs(z_alpha) * sigma_portfolio)
    # ES Paramétrico bajo normalidad
    es_param = inversion * (-mu_portfolio + sigma_portfolio * (norm.pdf(z_alpha) / (1-confianza)))

    # -- MÉTODO 2: SIMULACIÓN HISTÓRICA
    # Escenarios basados en retornos pasados sin suponer distribución
    rend_hist_port = rendimientos.dot(weights) # Rendimiento real del portfolio
    var_hist = inversion * (-np.percentile(rend_hist_port, (1 - confianza) * 100))
    es_hist = inversion * (-rend_hist_port[rend_hist_port <= (np.percentile(rend_hist_port, (1 - confianza) * 100))].mean())

    # -- MÉTODO 3: SIMULACIÓN DE MONTE CARLO
    # Genera escenarios futuros con un modelo estocástico (10000 sim)
    # Usamos descomposición de Cholesky para mantener correlaciones
    L = np.linalg.cholesky(matriz_cov)
    z = np.random.normal(size=(n_sim, len(tickers)))
    rend_simulados = mu_activos.values + z @ L.T
    rend_sim_port = rend_simulados @ weights

    var_mc = inversion * (-np.percentile(rend_sim_port, (1 - confianza) * 100))
    es_mc =inversion * (-rend_sim_port[rend_sim_port <= (np.percentile(rend_sim_port, (1 - confianza) * 100))].mean())

    return {
        "Paramétrico": (var_param, es_param),
        "Histórico": (var_hist, es_hist),
        "Monte Carlo": (var_mc, es_mc)
    }

# Ejecución para ambos periodos
res_pre = calcular_todo_el_riesgo(rend_activos.loc[inicio_pre:fin_pre], inversion_total, pesos, n_simulaciones)
res_pos = calcular_todo_el_riesgo(rend_activos.loc[inicio_pos:fin_pos], inversion_total, pesos, n_simulaciones)

# 3. Presentación de Resultados
for metodo in res_pre.keys():
    print(f"\n--- {metodo.upper()} ---")
    data = {
        "Métrica": ["VaR (95%)", "Expected Shortfall"],
        "Pre-Elecciones": [f"{res_pre[metodo][0]:.2f} €", f"{res_pre[metodo][1]:.2f} €"],
        "Pos-Elecciones": [f"{res_pos[metodo][0]:.2f} €", f"{res_pos[metodo][1]:.2f} €"]
    }
    print(pd.DataFrame(data).to_string(index=False))


--- PARAMÉTRICO ---
           Métrica Pre-Elecciones Pos-Elecciones
         VaR (95%)       800.71 €       372.94 €
Expected Shortfall      1002.67 €       497.77 €

--- HISTÓRICO ---
           Métrica Pre-Elecciones Pos-Elecciones
         VaR (95%)       801.06 €       251.56 €
Expected Shortfall       931.17 €       360.89 €

--- MONTE CARLO ---
           Métrica Pre-Elecciones Pos-Elecciones
         VaR (95%)       801.46 €       378.83 €
Expected Shortfall       997.81 €       493.58 €


### **Conclusiones del Análisis de Riesgo Multivariante (Netflix, Amaxon, Disney)**

A partir de la implementación de las tres metodologías cuantitativas sobre la cartera equiponderada (Netflix, Amazon y Disney) con una inversión de **24.000 €** y un nivel de confianza del **95%**, se extraen las siguientes observaciones clave:

#### **1. Impacto del Cambio de Régimen Político (Pre vs. Pos-Electoral)**
Se constata una **reducción drástica y generalizada del riesgo** en el periodo poselectoral. El VaR Paramétrico disminuye de **800,71 €** a **372,94 €**. Este comportamiento evidencia la resolución de incertidumbre: una vez despejada la incógnita política, la volatilidad se estabiliza, minorando la expectativa de pérdida máxima.

#### **2. Divergencia Muestral (Efecto Histórico)**
En el periodo preelectoral, los métodos Histórico (801,06 €) y Paramétrico (800,71 €) convergen casi perfectamente. Sin embargo, en el pos-electoral, el **VaR Histórico cae a 251,56 €**, mostrando un optimismo mucho mayor. Esto resalta la sensibilidad de la Simulación Histórica a la ventana de datos (45 días): si no ocurren shocks en ese lapso, el riesgo estimado cae abruptamente, mientras el paramétrico mantiene un suelo más conservador.

#### **3. Convergencia de Monte Carlo**
Al basarse en una distribución normal multivariante (vía Cholesky), los resultados de **Monte Carlo** convergen casi exactamente con el Modelo Paramétrico (ej. VaR Pre: 801,46 € vs. 800,71 €). Las  diferencias son resultado del ruido estadístico que  debería decrecer con el número de iteraciones.

#### **4. Beneficio de la Diversificación**
Activos como Netflix presentaban un VaR individual de **-1.266,96 €**. Al diversificar en una cartera equiponderada, el VaR agregado bajó a **800,71 €**. Esto demuestra el principio de **subaditividad**: la correlación imperfecta entre sectores diluye los riesgos individuales.


### **Evaluación Técnica Avanzada: Pros y Contras de las Metodologías**

Para profundizar en la evaluación técnica de cada enfoque operativo, se detallan a continuación las ventajas y desventajas metodológicas identificadas:

#### **1. Método Paramétrico (Varianza-Covarianza)**
*   **Pro:** **Velocidad de cómputo y transparencia analítica.** El cálculo se obtiene mediante una fórmula cerrada sin necesidad de generar simulaciones numéricas, lo que lo convierte en el método óptimo para el reporting rápido en tiempo real. Además, captura el efecto de diversificación de forma explícita a través de la matriz $\Sigma$.
*   **Pro:** Permite una **atribución analítica exacta (Component VaR)**. Al basarse en matrices algebraicas, se puede derivar matemáticamente el VaR Marginal, permitiendo a los gestores saber exactamente cuántos euros de riesgo aporta cada acción (como Disney o Amazon) al cómputo global.
*   **Contra:** **Subestimación de eventos extremos (Colas Gruesas).** Al asumir una distribución Normal, ignora la asimetría y las colas pesadas empíricas del mercado, arrojando estimaciones de riesgo excesivamente optimistas en periodos de volatilidad.
*   **Contra:** Presenta **vulnerabilidad ante la ruptura de correlación en colas**. Asume una correlación constante, pero en momentos de pánico extremo, las correlaciones tienden a aumentar hacia 1 (dependencia de cola), lo que este método no asimila, subestimando pérdidas sistémicas.

#### **2. Simulación Histórica**
*   **Pro:** **Enfoque no paramétrico y captura empírica.** No asume ninguna distribución teórica, lo que le permite capturar automáticamente las colas gruesas y asimetrías que efectivamente acontecieron en el mercado.
*   **Pro:** Logra una **mitigación total del riesgo de modelo**. Al prescindir de la calibración de parámetros complejos, se eliminan errores de especificación, reflejando fielmente los datos puros de mercado.
*   **Contra:** **Dependencia absoluta del pasado.** El modelo es incapaz de generar escenarios de crisis que no hayan ocurrido previamente en la ventana temporal elegida (es "ciego" ante eventos sin precedentes).
*   **Contra:** Sufre el impacto de los "Efectos Fantasma". Un shock financiero permanecerá alterando el VaR durante toda la ventana elegida. Cuando ese evento sale de la muestra, el VaR cae violentamente, induciendo al error de creer que el mercado es más seguro.

#### **3. Simulación de Monte Carlo**
*   **Pro:** **Máxima flexibilidad operativa.** Es ideal para carteras complejas con derivados no lineales. Al generar escenarios sintéticos, logra dibujar una cola extrema densa y continua, solucionando problemas de falta de datos.
*   **Pro:** Es ideal para el **análisis de escenarios condicionales**. Ofrece un entorno óptimo para realizar *Stress Testing*, simulando qué ocurriría ante shocks de volatilidad que nunca se han registrado en la historia real.
*   **Contra:** **Alto coste computacional.** Requiere la ejecución de miles de simulaciones para alcanzar estabilidad estadística, lo que demanda algoritmos optimizados para carteras de gran escala.
*   **Contra:** Tiene una alta **sensibilidad al generador pseudoaleatorio**. Si no se emplean algoritmos de alta calidad, la simulación puede sufrir de agrupamientos espurios o pérdida de uniformidad, introduciendo dependencias artificiales.

### Práctica 9: Sobre Opciones: Las "Griegas" utilizando Black-Scholes

Algunas de las métricas más significativas en el entorno de las opciones son **"las griegas"**.

Podríamos hacernos las siguientes preguntas:

* ¿Cuánto ganamos/perdemos ante movimientos en el activo subyacente?

* ¿Cuánto ganamos/perdemos ante movimientos de la volatilidad?

* ¿Cuánto ganamos/perdemos por el paso de un día?

A simple vista no podemos responder a estas preguntas. A partir de los diferentes parámetros que afectan al precio de una opción, se definen diversos indicadores que sí nos permitirán responder a diversas preguntas:


* Delta: variación de la prima ante variaciones de un tick en el activo subyacente.
* Gamma: variación de delta ante variaciones de un tick en el activo subyacente.
* Vega: variación de la prima ante variaciones del 1% en la volatilidad implícita.
* Theta: variación de la prima ante el paso de 1 día.
* Rho: variación de la prima ante la variación del 1% en el tipo de interés. Tiene poca repercusión.

#### Implementación de la fórmula de Black-Scholes

In [ ]:
import numpy as np
from scipy.stats import norm

#### Definimos los parámetros

In [ ]:
rc = 0.01
dc = 0.0
S = 30
K = 40
T = 240/365
sigma = 0.30

#### Definimos la función de blackScholes

Obtener el valor de una opción call usando la fórmula de Black-Scholes y análogamente para una opción put:  

$$
V_{\mathrm{call}}(S,t) = Se^{-d_c t} \Phi(d_1) - Ke^{-r_c t} \Phi(d_2),
$$

donde:
$$
d_1 = \frac{\ln(S/K) + (r_c -d_c + 0.5 \sigma^2)t}{\sigma \sqrt{t}}, \quad d_2 = d_1 -\sigma\sqrt{t}.
$$



In [ ]:
def blackScholes(rc, dc, S, K, T, sigma, type="c"):
    "Calculate BS price of call/put"
    d1 = (np.log(S/K) + (rc - dc + sigma**2/2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if type == "c":
        price = S*np.exp(-dc*T)*norm.cdf(d1, 0, 1) - K*np.exp(-rc*T)*norm.cdf(d2, 0, 1)
    elif type == "p":
        price = K*np.exp(-rc*T)*norm.cdf(-d2, 0, 1) - S*np.exp(-dc*T)*norm.cdf(-d1, 0, 1)
    return price

In [ ]:
print("Option Price: ", blackScholes(rc, dc, S, K, T, sigma, "c"))

Option Price:  0.5132843798399405


## Delta $ \Delta \delta$

Delta mide la sensibilidad de una opción a un cambio en el activo subyacente, esencialmente describiendo cuánto cambiará el precio de la opción para un movimiento dado en el precio del subyacente.

Las opciones Call tienen deltas positivos, lo que refleja el hecho de que su precio aumenta con un aumento en el precio del activo subyacente, en igualdad de condiciones. En consecuencia, las opciones Put tienen deltas negativos. En términos absolutos, el delta de una opción es un número que está limitado entre 0 y 1 (o 0 y 100 cuando se expresa en términos porcentuales), siendo el de las opciones at-the-money cercano a 0,5.

Delta también se considera comúnmente como una indicación de la probabilidad, en un momento dado, de que la opción caduque "in the money".

Desde un punto de vista matemático:

Delta mide la tasa de cambio del valor teórico de la opción con respecto al activo subyacente.

$ \Delta = \frac{\delta V}{\delta S} $

$ \Delta_{call} = \Phi (d1) $

$ \Delta_{put} = -\Phi (-d1) $

In [ ]:
def delta_calc(rc, dc, S, K, T, sigma, type="c"):
    "Calculate delta of an option"

    # 1. Calculo de d1
    d1 = (np.log(S / K) + (rc - dc + sigma**2 / 2) * T) / (sigma * np.sqrt(T))

    # 2.Fórmulas de Delta
    if type == "c":
        # Delta de una Call
        delta = norm.cdf(d1, 0, 1)
    elif type == "p":
        # Delta de una Put
        delta = -norm.cdf(-d1, 0, 1)

    return delta

## Gamma $ \Gamma \gamma $

Delta solo es útil para determinar la sensibilidad de una opción a pequeños movimientos en el subyacente, porque no se mantiene constante. El parámetro de riesgo de opción que mide la tasa por la cual el delta cambia con los cambios en el precio subyacente se llama gamma.

Las opciones largas tienen gamma positivo, mientras que las opciones cortas tienen negativo. Gamma está en su punto más alto para las opciones at-the-money y próximas al vencimiento.

Gamma mide la tasa de cambio en el delta con respecto a los cambios en el precio subyacente.


$ \Gamma = \frac{\delta \Delta}{\delta S} = \frac{\delta V}{\delta S^2} $


$ \Gamma = \frac {\Phi ^{'} (d1)}{S \sigma \sqrt{\tau}} $



In [ ]:
def gamma_calc(rc, dc, S, K, T, sigma, type="c"):
    "Calculate gamma of an option"

    # 1. Calculo de d1
    d1 = (np.log(S/K) + (rc - dc + sigma**2 / 2) * T) / (sigma*np.sqrt(T))

    # 2. Fórmula de Gamma
    # Gamma = Φ'(d1) / (S*sigma*sqrt(T))
    # norm.pdf(d1, 0, 1) representa la derivada de la función de distribución (Φ')
    gamma = norm.pdf(d1, 0, 1) / (S * sigma * np.sqrt(T))

    return gamma

## Vega $ \nu $

Aunque vega no es en realidad una letra griega, es una de las "griegas" más importantes en el comercio de opciones; mide la sensibilidad de la opción a cambios en su volatilidad implícita, típicamente para un movimiento del 1% en esta última.

Al igual que con gamma, las opciones largas tienen vega positivo, mientras que las opciones cortas tienen negativo. Sin embargo, al contrario del caso de gamma, cuanto más tiempo quede hasta el vencimiento de la opción, mayor será su vega. Vega es, por lo tanto, una de las consideraciones de riesgo importantes para las opciones a más largo plazo.

Vega mide la sensibilidad a la volatilidad. Vega es el derivado del valor de la opción con respecto a la volatilidad del activo subyacente.

$ \nu = \frac{\delta V}{\delta \sigma} $


$ \nu = S \Phi ^{'} (d1) \sqrt{\tau} $

In [ ]:
def vega_calc(rc, dc, S, K, T, sigma, type="c"):
    "Calculate vega of an option"

    # 1. Calculo de d1
    d1 = (np.log(S/K) + (rc - dc + 0.5 * sigma**2) * T)/(sigma*np.sqrt(T))

    # 2. Fórmula de Vega
    # nu = S * Φ'(d1) * sqrt(T)
    vega = S * norm.pdf(d1, 0, 1) * np.sqrt(T)/100

    return vega

## Theta $ \Theta \theta $

Theta es el parámetro de riesgo utilizado para describir la influencia del tiempo en el valor de la opción. Por lo general, se expresa como la pérdida que sufrirá la opción durante un día calendario. Por esa razón, las opciones a menudo se cotizan a la baja los viernes, para incorporar los efectos del valor del tiempo de tres días a punto de perderse.

Theta puede considerarse como una compensación por gamma. Las opciones largas tienen theta negativo y las opciones cortas tienen positivo.

Theta mide la sensibilidad del valor de la opción al paso del tiempo - decaimiento del tiempo.

$ \Theta = -\frac{\delta V}{\delta \tau} $


$ \Theta_{call} = -\frac {S \Phi ^{'} (d1) \sigma}{2\sqrt{\tau}} -rK \exp(-rT)\Phi(d_2) $

$ \Theta_{put} = -\frac {S \Phi ^{'} (d1) \sigma}{2\sqrt{\tau}} +rK \exp(-rT)\Phi(-d_2) $

In [ ]:
def theta_calc(rc, dc, S, K, T, sigma, type="c"):
    "Calculate theta of an option"

    # 1. Calculo de d1 y d2
    d1 = (np.log(S/K) + (rc-dc + 0.5*sigma**2) * T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # 2. Término común
    # Term1 = -(S*Φ'(d1)*sigma)/(2*sqrt(T))
    term1 = -(S*norm.pdf(d1, 0, 1)*sigma) / (2*np.sqrt(T))

    # 3. Implementación para Call y Put
    if type == "c":
        # Theta_call = Term1 - r*K*exp(-rT)*Φ(d2)
        theta = (term1 - rc*K*np.exp(-rc*T)*norm.cdf(d2, 0, 1))/365
    elif type == "p":
        # Theta_put = Term1 + r*K*exp(-rT)*Φ(-d2)
        theta = (term1 + rc*K*np.exp(-rc*T)*norm.cdf(-d2, 0, 1))/365

    return theta

## Rho $ \rho $

Rho generalmente se considera la menos importante de las griegos más seguidas. Describe la sensibilidad de una opción a los cambios en la tasa de interés utilizada para derivar el valor de la opción.

Un aumento en las tasas de interés aumentará el precio a plazo del subyacente y, por lo tanto, será bueno para las opciones Call y malo para las opciones Put. Las primeras tienen por tanto rho positivo, mientras que las segundas negativo.

Rho mide la sensibilidad a la tasa de interés.

$ \rho = -\frac{\delta V}{\delta r} $


$ \rho_{call} = K\tau \exp(-rT)\Phi(d_2) $

$ \rho_{put} = - K\tau \exp(-rT)\Phi(d_2) $

In [ ]:
def rho_calc(rc, dc, S, K, T, sigma, type="c"):
    "Calculate rho of an option"

    # 1. Calculo de d1 y d2
    d1 = (np.log(S/K) + (rc-dc + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    # 2. Fórmulas de Rho
    if type == "c":
        # Rho de una Call: K*T*exp(-rT)*Φ(d2)
        rho = K * T * np.exp(-rc * T) * norm.cdf(d2, 0, 1)/100
    elif type == "p":
        # Rho de una Put: -K*T*exp(-rT)*Φ(-d2)
        rho = -K * T * np.exp(-rc * T) * norm.cdf(-d2, 0, 1)/100

    return rho

In [ ]:
option_type= ['c', 'p']

print("Option Price: ", [round(blackScholes(rc, dc, S, K, T, sigma, x),3) for x in option_type])
print("       Delta: ", [round(delta_calc(rc, dc, S, K, T, sigma, x),3) for x in option_type])
print("       Gamma: ", [round(gamma_calc(rc, dc, S, K, T, sigma, x),3) for x in option_type])
print("       Vega : ", [round(vega_calc(rc, dc, S, K, T, sigma, x),3) for x in option_type])
print("       Theta: ", [round(theta_calc(rc, dc, S, K, T, sigma, x),3) for x in option_type])
print("       Rho  : ", [round(rho_calc(rc, dc, S, K, T, sigma, x),3) for x in option_type])

Option Price:  [0.513, 10.251]
       Delta:  [0.151, -0.849]
       Gamma:  [0.032, 0.032]
       Vega :  [0.057, 0.057]
       Theta:  [-0.004, -0.003]
       Rho  :  [0.026, -0.235]


<font size="2" color=grey> Nota para comprobación, Opción tipo Call:
    
    * Valor:  [0.513]   
    * Delta:  [0.151]
    * Gamma:  [0.032]
    * Vega :  [0.057]
    * Theta:  [-0.004]
    * Rho  :  [0.026]

De las salidas de las dos celdas de código anteriores se comprueba que los resultados obtenidos con las funciones defindas coinciden con los valores dados para su comprobación para una opción tipo Call. A continuación podemos responder a las preguntas planteadas en el enunciado a las cuales no se había encontrado una primera respuesta.

A partir de los resultados obtenidos en el modelo de Black-Scholes para los parámetros definidos ($S=30, K=40, T=240/365, \sigma=0.30, r_c=0.01$), damos respuesta a las preguntas planteadas sobre la sensibilidad de la cartera:

**1. ¿Cuánto ganamos/perdemos ante movimientos en el activo subyacente? (Delta y Gamma)**
* **Delta ($\Delta$):** Mide el cambio inmediato en la prima ante una variación de +1 unidad en el precio del subyacente ($S$ pasa de 30 a 31). Si esto ocurre:
  * La prima de la Call aumentará en 0.151 unidades. Gana valor porque se acerca a su *Strike*.
  * La prima de la Put disminuirá en 0.849 unidades. Pierde valor porque al subir el mercado, vender a 40 pierde atractivo.

  * Nota teórica: Se cumple con exactitud la relación de paridad en ausencia de dividendos: $\Delta_{Call} - \Delta_{Put} = 1.00$ ($0.151 - (-0.849) = 1$).
  * *Probabilidades:* El mercado asigna un ~15% de probabilidad de que la Call termine ITM, frente a un ~85% para la Put.
* **Gamma ($\Gamma$)**:Es idéntica para ambas opciones por propiedades simétricas del modelo. Si el subyacente sube 1 unidad, la Delta de la Call se "acelera" hasta 0.183 ($0.151 + 0.032$) y la Delta de la Put se vuelve menos negativa, situándose en -0.817 ($-0.849 + 0.032$). Gamma beneficia al comprador en ambos escenarios.

**2. ¿Cuánto ganamos/perdemos ante movimientos de la volatilidad? (Vega)**
* **Vega ($\mathcal{V}$)**: Mide el impacto de un incremento del 1% en la volatilidad implícita ($\sigma$ pasa de 30% a 31%).Es idéntica para ambas opciones (**0.057**). Si la volatilidad implícita sube un 1%, tanto para la Call como para la Put, las primas se encarecerán en 0.057. Al ser compradores de opciones, nos beneficia la incertidumbre y los movimientos bruscos, ya que ensancha las colas de la distribución de probabilidad, aumentando la opción de valor al vencimiento sin añadir riesgo de pérdida más allá de la prima pagada.

**3. ¿Cuánto ganamos/perdemos por el paso de un día? (Theta)**
* **Theta ($\Theta$)**:Representa el decaimiento temporal diario, asumiendo que el precio del subyacente y la volatilidad permanecen constantes. Bajo esta situación, el paso de un día natural nos hará perder dinero en ambas opciones:

  * La Call pierde **-0.004** unidades de valor por cada día natural que transcurre.
  * La Put pierde **-0.003** unidades de valor por día.
  
  El tiempo corre en contra del comprador de la opción.

**4. Impacto de los tipos de interés (Rho)**
* **Rho ($\rho$):** Mide la sensibilidad de la opción ante un incremento del 1% en la tasa libre de riesgo ($r_c$ pasa de 1% a 2%).
  * Opción Call ($\rho = +0.026$): La prima aumenta 0.026 unidades. Un tipo de interés más alto reduce el valor presente del precio de ejercicio ($K$) que se pagaría en el futuro.
  * Opción Put ($\rho = -0.235$): La prima disminuye 0.235 unidades. Un tipo de interés más alto penaliza fuertemente el valor actual del efectivo ($K=40$) que el comprador espera recibir en el futuro al vender el activo.

ASimismo, podemos comparar los resultados obtenidos con los que nos proporciona la librería de Python ``vollib``.

In [ ]:
#!pip install vollib

In [ ]:
import vollib
from vollib.black_scholes import black_scholes
from vollib.black_scholes.greeks.analytical import delta, gamma, vega, theta, rho

In [ ]:
import numpy as np
from scipy.stats import norm

**Definimos los parámetros**

In [ ]:
rc=0.01
dc=0.0
S=30
K=40
T=240/365
sigma=0.30


In [ ]:
option_types = ['c', 'p']
data = []

for opt in option_types:
    # Funciones definidas en este notebook
    p_mine = blackScholes(rc, dc, S, K, T, sigma, opt)
    d_mine = delta_calc(rc, dc, S, K, T, sigma, opt)
    g_mine = gamma_calc(rc, dc, S, K, T, sigma, opt)
    v_mine = vega_calc(rc, dc, S, K, T, sigma, opt)
    t_mine = theta_calc(rc, dc, S, K, T, sigma, opt)
    r_mine = rho_calc(rc, dc, S, K, T, sigma, opt)

    # Librería Vollib
    p_vol = black_scholes(opt, S, K, T, rc, sigma)
    d_vol = delta(opt, S, K, T, rc, sigma)
    g_vol = gamma(opt, S, K, T, rc, sigma)
    v_vol = vega(opt, S, K, T, rc, sigma)
    t_vol = theta(opt, S, K, T, rc, sigma)
    r_vol = rho(opt, S, K, T, rc, sigma)

    # Resultados
    label = "Call" if opt == 'c' else "Put"
    metrics = [
        ("Precio", p_mine, p_vol), ("Delta", d_mine, d_vol),
        ("Gamma", g_mine, g_vol), ("Vega", v_mine, v_vol),
        ("Theta", t_mine, t_vol), ("Rho", r_mine, r_vol)]

    for m_name, m_mine, m_vol in metrics:
        data.append([label, m_name, round(m_mine, 4), round(m_vol, 4)])

df_comp = pd.DataFrame(data, columns=["Tipo", "Métrica", "Mis Funciones", "Vollib"])
print(df_comp)

    Tipo Métrica  Mis Funciones   Vollib
0   Call  Precio         0.5133   0.5133
1   Call   Delta         0.1506   0.1506
2   Call   Gamma         0.0320   0.0320
3   Call    Vega         0.0569   0.0569
4   Call   Theta        -0.0037  -0.0037
5   Call     Rho         0.0263   0.0263
6    Put  Precio        10.2511  10.2511
7    Put   Delta        -0.8494  -0.8494
8    Put   Gamma         0.0320   0.0320
9    Put    Vega         0.0569   0.0569
10   Put   Theta        -0.0026  -0.0026
11   Put     Rho        -0.2350  -0.2350


### Conclusión de la Validación

Tras comparar las salidas de nuestras funciones manuales con los valores de referencia y la librería `vollib`, se confirma que:

1.  **Coincidencia:** Los resultados obtenidos mediante la librería `vollib` coinciden con los obtenidos a través de las funciones implementadas manualmente, tanto para la opción **Call** como para la opción **Put**.
2.  **Precisión:** Los valores para la opción **Call** coinciden con los proporcionados en la nota de comprobación (Precio: 0.513, Delta: 0.151, etc.).
3.  **Robustez:** La implementación es válida para su uso en análisis de sensibilidad de opciones financieras dentro del modelo Black-Scholes.